# Energy budget closure across the model components

Where does EC-Earth4 create or lose energy? The budget follows the energy from TOA through three parts, each with its residual ε (W m⁻², **> 0: energy created**):

| part | residual | definition |
|---|---|---|
| OIFS / HTESSEL | `oifs_res` = `atm_imb` + `land_res` | surface − TOA; calving out − land flux in (land storage ≈ 0) |
| OASIS band | `band_res` = `oasis_res` + `snow_res` + `water_cycle` | out − in for each flux crossing the coupler: heat, snow on ice / open water, evaporation + rain, calving, runoff |
| NEMO / SI3 | `oce_res` | heat uptake (SI3 storage + OHC tendency) − inputs |
| total | `total_res` | heat uptake − TOA − geothermal = sum of the three |

Formulas, required output, checks and the reasons for each choice: `ENERGY_BUDGET.md`. Runs with standard output only get the total and the combined residuals (`band_res_std`, `oce_imb`).

In [ ]:
import check_run as cr
import energy_budget as eb
from importlib import reload
import os
reload(cr); reload(eb)

# runs: (exp, user)
runs = [('xax0', 'ccpd'),     # ORCA2, OIFS 3600 / NEMO 5400 s (as xc09)
        ('ACX0', 'ccpd'),     # PALEORCA2, eocene OIFS 2700 / NEMO 3600 s (as AC01)
        ('AC01', 'ccpd'),     # PALEORCA2, eocene OIFS 2700 / NEMO 3600 s: standard output only (same simulation as ACX0)
        ('aI09', 'ccff')]     # TL255-eORCA1, OIFS 1800 / NEMO 2700 s, nn_fsbc=2: standard output only (no water heat terms)
exps, users = [r[0] for r in runs], [r[1] for r in runs]

exp_labels = {
    'xax0': {'label': 'ORCA2, OIFS 3600 / NEMO 5400 s', 'color': 'navy'},
    'ACX0': {'label': 'PALEORCA2, OIFS 2700 / NEMO 3600 s', 'color': 'darkorange'},
    'AC01': {'label': 'PALEORCA2, OIFS 2700 / NEMO 3600 s, standard output', 'color': 'goldenrod'},
    'aI09': {'label': 'TL255-eORCA1, OIFS 1800 / NEMO 2700 s, standard output', 'color': 'seagreen'},
}
read_again = ['xax0', 'ACX0']                   # still running: new years are appended (aI09: saved files only)

cart_exp = '/ec/res4/scratch/{}/ece4/'
cart_out = '/etc/ecmwf/nfs/dh1_perm_b/ccpd/epochal/analysis'
cart_figs = cart_out + '/budget/'
os.makedirs(cart_figs, exist_ok = True)

# years averaged in the tables and diagrams: positions in each run's series, (1, 10) = years 2-10;
# or 'last 20' for the last 20 years
window = (1, 10)

# units of the tables and of the figure: 'global' = contributions to the global mean (W m-2 per Earth m2, all the
# numbers add up and the total budget closes); 'region' = each term per unit area of its own region
units = 'global'

## Read the runs

`check_run.read_output` reads (or updates) the saved annual global means and applies the ocean and coupling budget (`compute_interface_balance`); `read_water_heat` reads the water heat terms of the runs that write them (cached in `exps_clim/budget_water_<exp>.nc`). Runs without a complete year yet are skipped with a warning.

In [ ]:
clim_all = cr.read_output(exps, user = users, read_again = read_again, cart_exp = cart_exp,
                          cart_out = cart_out + '/exps_clim/',
                          atmvars = eb.ATMVARS, ocevars = eb.OCEVARS, icevars = eb.ICEVARS)
# water heat terms over the same years as the saved means (last_year refreshes the cache when it is behind)
last_year = lambda exp: int(clim_all['oce_mean'][exp].year[-1]) if clim_all.get('oce_mean', {}).get(exp) is not None else None
water = {exp: eb.read_water_heat(exp, us, cart_exp = cart_exp, cart_out = cart_out + '/exps_clim/', last_year = last_year(exp))
         for exp, us in zip(exps, users)}
terms = eb.budget_terms(clim_all, exps, users, water = water, cart_exp = cart_exp)

desc, colors_by_exp, _ = cr.parse_exp_labels(exp_labels)
ok = [e for e in exps if e in terms]                            # runs with data
colors = dict(zip(exps, cr.resolve_colors(exps, colors_by_exp)))
labels = {e: f'{e}: {desc[e]}' if e in desc else e for e in exps}
print('runs with data:', ok, '| with water heat terms:', [e for e in ok if water.get(e) is not None])

## Budget table

Mean of each term over `window` (W m⁻²). With `units = 'global'` every row is a contribution to the global mean (per Earth area), so the rows can be summed; with `units = 'region'` each row is per unit area of its own region.

In [ ]:
tab = eb.budget_table(terms, window = window, units = units)
show = ['years', 'toa', 'srf_net', 'atm_imb', 'srf_land_glob', 'cal_latent', 'land_res', 'F_oifs', 'qt_atm_oi', 'qemp', 'interface_imb', 'oasis_res', 'water_cycle',
        'ice_storage', 'qt_oce_ai', 'runoff_heat', 'F_geo', 'ohc_trend', 'oce_imb', 'oce_res', 'end_to_end', 'heat_uptake', 'total_res']
display(tab[show].style.format({c: '{:+.3f}' for c in show if c != 'years'}, na_rep = 'n/a'))
tab.to_csv(cart_figs + f"budget_table_{'-'.join(ok)}_{units}.csv")

### Closure checks

One check per box (`eb.CHECKS`, same list as in `ENERGY_BUDGET.md`): the signed sum of its terms must be 0 within the tolerance. Tolerance 1e-6: identity by construction (code and definitions agree, all terms over the same years). Larger tolerance: real check between independent estimates (surface split with two weightings; NEMO's E − P heat flux against its separate diagnostics); the "Total" checks repeat the surface-split difference. Checks that need terms a run does not have are left empty.

In [ ]:
checks = eb.budget_checks(eb.budget_table(terms, window = window, units = 'global'), exps = ok)   # checks need global units
okcols = [f'{e} ok' for e in ok]
display(checks[['expression', 'tolerance'] + ok + okcols + ['note']].style
        .format({e: '{:+.1e}' for e in ok} | {'tolerance': '{:.0e}'}, na_rep = 'n/a')
        .map(lambda v: 'background-color: #d4edda' if v is True else ('background-color: #f8d7da' if v is False else ''), subset = okcols))
print('all available checks passed' if checks[okcols].fillna(True).all().all() else 'SOME CHECKS FAILED')

## The flow, frame by frame

One row per run: TOA enters the OIFS / HTESSEL frame from the left; the ocean-point flux, the snowfall (on ice and on open water), the water (mass only) and the calving cross the OASIS band, which shows for each flux what OIFS gives, what NEMO / SI3 books and ε = out − in; the outputs are summed into the SI3 sea-ice and NEMO ocean boxes of the NEMO / SI3 frame; the total ε is on the right. Fluxes are rounded to 0.01 and every residual is computed from the rounded fluxes, so each box adds up in the digits shown.

In [ ]:
fig = eb.plot_budget_flow(tab, exps = ok, labels = labels, cart_out = cart_figs)

In [ ]:
fig = eb.plot_residuals(tab, exps = ok, colors = [colors[e] for e in ok], labels = labels, cart_out = cart_figs)

## Time series

Annual values (5-year rolling mean): are the residuals steady, or do they drift with the climate (sea-ice melt, ocean warming)?

In [ ]:
fig = eb.plot_terms_ts(terms, exps = ok, colors = [colors[e] for e in ok], labels = labels, rolling = 5, cart_out = cart_figs)

In [ ]:
fig = eb.plot_terms_ts(terms, variables = ['F_oifs', 'qt_atm_oi', 'qemp', 'ice_storage', 'qt_oce_ai', 'runoff_heat', 'ohc_trend'],
                       exps = ok, colors = [colors[e] for e in ok], labels = labels, rolling = 5)

## The freshwater cycle test (runs with the water heat terms)

If the hypothesis holds: `oasis_res` ≈ 0 (OASIS + NEMO's additions explain what NEMO receives), `oce_res` ≈ 0 (the ocean residual is the runoff heat) and `qemp_sens` ≈ −`runoff_heat` (the water leaves and comes back with the same heat content). The split of `qemp_oce` into evaporation, rain and snow shows how much the temperature given to the precipitation (rain at SST) matters compared with net evaporation.

In [ ]:
wcols = ['years', 'F_oifs_nosn', 'snow_melt', 'qt_atm_oi', 'qemp', 'qemp_sens', 'cal_latent', 'interface_imb', 'oasis_res',
         'oce_imb', 'runoff_heat', 'oce_res', 'end_to_end', 'qemp_oce', 'qemp_ice', 'hflx_evap_cea', 'hflx_rain_cea', 'hflx_snow_cea']
wexps = [e for e in ok if water.get(e) is not None]
if wexps:
    wtab = eb.budget_table({e: terms[e] for e in wexps}, window = window, columns = wcols[1:])
    display(wtab[wcols].style.format({c: '{:+.3f}' for c in wcols if c != 'years'}, na_rep = 'n/a'))
    fig = eb.plot_terms_ts({e: terms[e] for e in wexps}, variables = ['interface_imb', 'qemp_sens', 'oasis_res', 'oce_imb', 'runoff_heat', 'oce_res'],
                           exps = wexps, colors = [colors[e] for e in wexps], labels = labels)
else:
    print('no run with the water heat terms yet (xax0 and ACX0 need a complete year)')

## Notes for the report

Questions the numbers should answer once xax0 and ACX0 have 10 years:
1. Is `oasis_res` ≈ 0 and `oce_res` ≈ 0? Then the ±0.4 W m⁻² pair is the freshwater heat content, not a leak. If `oasis_res` stays ~+0.3 (first year of xax0), where does it come from: OASIS remapping (heat fluxes without global conservation), or the dQ/dT correction SI3 applies to the received ice flux? Splitting it over ice and open water needs the OIFS tile fluxes.
1b. Where does the calving/iceberg latent heat (−0.08 in xax0) come from, with no calving field in namcouple?
2. Is the split of `qemp` dominated by net evaporation (`hflx_evap_cea`), or does the rain temperature (`hflx_rain_cea`) matter?
3. Do the terms differ between ORCA2 (3600/5400 s) and PALEORCA2 (2700/3600 s) beyond the atmosphere residual (OIFS time step)?
4. What does the −0.1 W m⁻² end-to-end residual of AA06 (3600/3600 s) correspond to? A diagnostic copy of AA06 would split it the same way.